# Comprehensive Machine Learning Reference Notebook (Deep Dive)

This notebook provides a deep mathematical and computational analysis of essential ML algorithms, focusing on:
- Supervised Learning (Regression & Classification)
- Unsupervised Learning (Clustering & Dimensionality Reduction)
- Detailed Complexity Analysis with Mathematical Derivations
- Bilingual Explanations

## 1. Supervised Learning: Regression

### 1.1 Linear Regression (OLS) - Ordinary Least Squares

**Mathematical Formulation:**
We aim to minimize the Sum of Squared Errors (SSE):
$$ J(\beta) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = ||Y - X\beta||^2 $$
Taking the derivative with respect to $\beta$ and setting it to zero yields the closed-form solution:
$$ \hat{\beta} = (X^T X)^{-1} X^T Y $$


**Complexity Analysis (Detailed Derivation):**
Let $n$ be the number of samples and $d$ be the number of features.

1.  **Matrix Multiplication $X^T X$:**
    - $X$ is of shape $(n, d)$. $X^T$ is $(d, n)$.
    - Resulting matrix is $(d, d)$.
    - Operations: Each of the $d^2$ elements requires $n$ multiplications and $n-1$ additions.
    - **Time:** $O(n \cdot d^2)$.

2.  **Matrix Inversion $(X^T X)^{-1}$:**
    - Inverting a $d \times d$ matrix using Gaussian elimination or Cholesky decomposition.
    - Standard matrix inversion is cubic in dimensions.
    - **Time:** $O(d^3)$.

3.  **Matrix Multiplication $(X^T X)^{-1} (X^T Y)$:**
    - $(d, d) \times (d, 1)$.
    - **Time:** $O(d^2)$.

**Total Time Complexity:** $O(n \cdot d^2 + d^3)$.
**Space Complexity:** $O(n \cdot d)$ to store the dataset, $O(d^2)$ for the covariance matrix.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression

# Generate data
X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)

# Fit model
lr = LinearRegression()
lr.fit(X, y)

print(f"R^2 Score: {lr.score(X, y):.4f}")
print(f"Coefficients shape: {lr.coef_.shape}")

R^2 Score: 1.0000
Coefficients shape: (10,)


### 1.2 Logistic Regression (Sigmoid & Cross-Entropy Loss)

**Mathematical Formulation:**
Predicts probability using the sigmoid function:
$$ h_\theta(x) = \frac{1}{1 + e^{-\theta^T x}} $$
Loss Function (Binary Cross-Entropy):
$$ J(\theta) = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(h_\theta(x_i)) + (1-y_i) \log(1-h_\theta(x_i))] $$
Optimized using Gradient Descent (no closed-form solution).



**Complexity Analysis (Gradient Descent):**
Let $k$ be the number of iterations.
1.  **Forward Pass (Prediction):** $\theta^T x$ takes $O(d)$ per sample. For $n$ samples: $O(n \cdot d)$.
2.  **Gradient Calculation:** Summing errors over $n$ samples takes $O(n \cdot d)$.
3.  **Update Step:** $O(d)$.

**Total Time Complexity:** $O(k \cdot n \cdot d)$.
**Space Complexity:** $O(n \cdot d)$ for data, $O(d)$ for weights.


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

# Generate classification data
X_cls, y_cls = make_classification(n_samples=1000, n_features=10, n_classes=2, random_state=42)

# Fit model
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_cls, y_cls)

print(f"Accuracy: {log_reg.score(X_cls, y_cls):.4f}")

Accuracy: 0.8590


### 1.3 Support Vector Machines (SVM) - Linear & Kernel

**Mathematical Formulation (Primal Problem):**
Minimize:
$$ \frac{1}{2} ||w||^2 $$
Subject to:
$$ y_i(w^T x_i + b) \geq 1, \quad \forall i $$
This is a Convex Quadratic Programming (QP) problem.

**Dual Problem (Kernel Trick):**
Using Lagrange multipliers $\alpha_i$:
$$ W(\alpha) = \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \sum_{i=1}^{n} \sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j K(x_i, x_j) $$
Where $K(x_i, x_j)$ is the kernel function.


**Complexity Analysis:**
1.  **Training:** Solving the QP problem. Standard solvers (like SMO - Sequential Minimal Optimization) typically take:
    - **Time:** $O(n^2 \cdot d)$ to $O(n^3 \cdot d)$. In worst case, it depends on the number of support vectors.
    - Why $O(n^2)$? The dual problem involves a kernel matrix of size $n \times n$. Calculating gradients or updates often involves operations on this matrix.
2.  **Prediction:** Depends only on Support Vectors ($N_{SV}$).
    - **Time:** $O(N_{SV} \cdot d)$. Since $N_{SV} \ll n$, prediction is fast.



In [3]:
from sklearn.svm import SVC

# Fit SVM
svm = SVC(kernel='rbf', C=1.0, gamma='scale')
svm.fit(X_cls, y_cls)

# Count support vectors
print(f"Number of Support Vectors: {svm.n_support_}")
print(f"Accuracy: {svm.score(X_cls, y_cls):.4f}")

Number of Support Vectors: [204 202]
Accuracy: 0.8950


### 1.4 K-Nearest Neighbors (KNN)

**Concept:** A non-parametric method that classifies an object based on the majority class of its $k$ nearest neighbors in the feature space.


**Complexity Analysis:**
1.  **Training:** $O(1)$ - KNN is a "lazy learner"; it stores all data.
2.  **Prediction:** For a single query point:
    - Calculate distance to all $n$ training points: $O(n \cdot d)$ (using Euclidean distance).
    - Sort distances or find top $k$: $O(n \log n)$ or $O(n)$ using selection algorithms.
    - **Total Time per Query:** $O(n \cdot d + n \log n)$.
    - **Optimization:** Using KD-Trees or Ball-Trees reduces average case to $O(d \cdot n^{1 - 1/d})$ but worst case remains $O(n \cdot d)$.


In [4]:
from sklearn.neighbors import KNeighborsClassifier

# Fit KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_cls, y_cls)

print(f"Accuracy: {knn.score(X_cls, y_cls):.4f}")

Accuracy: 0.8810


### 1.5 Naive Bayes (Gaussian)

**Mathematical Formulation:**
Based on Bayes' Theorem with the "naive" assumption of conditional independence:
$$ P(y|x_1, ..., x_d) = \frac{P(y) \prod_{i=1}^{d} P(x_i|y)}{P(x_1, ..., x_d)} $$
For Gaussian NB, $P(x_i|y)$ is modeled by a Normal distribution:
$$ P(x_i|y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} e^{-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}} $$


**Complexity Analysis:**
1.  **Training:** Calculate mean ($\mu$) and variance ($\sigma^2$) for each feature in each class.
    - **Time:** $O(n \cdot d)$. We iterate through all samples once.
2.  **Prediction:** Calculate likelihood for each class and multiply.
    - **Time:** $O(d \cdot C)$ where $C$ is number of classes. Since $C$ is usually small, it's $O(d)$.

In [5]:
from sklearn.naive_bayes import GaussianNB

# Fit Naive Bayes
gnb = GaussianNB()
gnb.fit(X_cls, y_cls)

print(f"Accuracy: {gnb.score(X_cls, y_cls):.4f}")

Accuracy: 0.8470


### 1.6 Decision Trees & Random Forests

**Concept:**
- **Decision Tree:** Recursively splits data based on feature thresholds to maximize Information Gain (or Gini Impurity).
- **Random Forest:** Ensemble of Decision Trees trained on bootstrap samples with random feature subsets.

**Complexity Analysis:**
1.  **Single Decision Tree Training:**
    - At each node, we evaluate all $d$ features and $O(n)$ split points (or $O(n \log n)$ if sorted).
    - Tree depth is $O(\log n)$ on average.
    - **Time:** $O(d \cdot n^2)$ in worst case (unpruned), $O(d \cdot n \log n)$ on average.
2.  **Random Forest Training:**
    - Build $t$ trees.
    - **Time:** $O(t \cdot d \cdot n \log n)$.



In [6]:
from sklearn.ensemble import RandomForestClassifier

# Fit Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_cls, y_cls)

print(f"Accuracy: {rf.score(X_cls, y_cls):.4f}")

Accuracy: 1.0000


## 2. Unsupervised Learning

### 2.1 K-Means Clustering

**Mathematical Formulation:**
Minimize the Intra-Cluster Sum of Squared Errors (SSE):
$$ J = \sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2 $$
Algorithm:
1. Initialize $k$ centroids randomly.
2. Assign each point to the nearest centroid.
3. Update centroids as the mean of assigned points.
4. Repeat until convergence.

**Complexity Analysis:**
Let $n$ be samples, $d$ be dimensions, $k$ be clusters, $i$ be iterations.
1.  **Assignment Step:** Calculate distance from each of $n$ points to $k$ centroids.
    - Distance calculation: $O(d)$.
    - Total: $O(n \cdot k \cdot d)$.
2.  **Update Step:** Calculate mean for $k$ clusters.
    - Total: $O(n \cdot d)$.
**Total Time Complexity:** $O(i \cdot n \cdot k \cdot d)$.
**Space Complexity:** $O(n \cdot d)$ to store data, $O(k \cdot d)$ for centroids.



In [7]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# Generate cluster data
X_cluster, y_cluster = make_blobs(n_samples=1000, centers=3, n_features=5, random_state=42)

# Fit K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_cluster)

print(f"Inertia: {kmeans.inertia_:.2f}")
print(f"Labels shape: {kmeans.labels_.shape}")

Inertia: 4868.83
Labels shape: (1000,)


### 2.2 DBSCAN (Density-Based Spatial Clustering)

**Concept:** Groups together points that are closely packed together (points with high density), marking as outliers points that lie alone in low-density regions.

**Complexity Analysis:**
1.  **Naive Implementation:**
    - For each point, find neighbors within radius $\epsilon$. This requires checking distance to all other points.
    - **Time:** $O(n^2 \cdot d)$.
2.  **Optimized (using KD-Tree/Ball-Tree):**
    - Range query takes $O(d \cdot n^{1 - 1/d})$ on average.
    - **Time:** $O(n \cdot d \cdot n^{1 - 1/d})$.


In [8]:
from sklearn.cluster import DBSCAN

# Fit DBSCAN
db = DBSCAN(eps=0.5, min_samples=5)
db.fit(X_cluster)

print(f"Number of clusters: {len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)}")
print(f"Noise points: {sum(db.labels_ == -1)}")

Number of clusters: 0
Noise points: 1000


### 2.3 Principal Component Analysis (PCA)

**Mathematical Formulation:**
Find orthogonal axes (principal components) that maximize variance. This is equivalent to finding the eigenvectors of the covariance matrix $\Sigma$.
$$ \Sigma = \frac{1}{n-1} X^T X $$
Solve: $\Sigma v = \lambda v$

**Complexity Analysis:**
1.  **Covariance Matrix:** $O(n \cdot d^2)$.
2.  **Eigendecomposition:** For a $d \times d$ matrix, $O(d^3)$.
    - **Total Time:** $O(n \cdot d^2 + d^3)$.
    - **Space:** $O(d^2)$.


In [9]:
from sklearn.decomposition import PCA

# Fit PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster)

print(f"Explained Variance: {pca.explained_variance_ratio_}")

Explained Variance: [0.82062307 0.15651775]


## 3. Deep Learning (Brief Overview)


### 3.1 Feedforward Neural Network (FNN)

**Complexity:** $O(L \cdot N_{neurons}^2)$ per forward pass.


In [10]:
import torch
import torch.nn as nn

class SimpleFNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleFNN, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x

fnn = SimpleFNN(10, 20, 2)
print(f"FNN Parameters: {sum(p.numel() for p in fnn.parameters())}")

FNN Parameters: 262


## Conclusion

This notebook provides a robust foundation for understanding the mathematical underpinnings and computational costs of classical ML algorithms. For deep learning, while the complexity is higher, the representational power is significantly greater.

